In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251118_141943"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "12_data_collection_multisource"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# 2. Importação
import pandas as pd
import numpy as np
import requests
from datetime import datetime
import time

In [4]:
# 3. Configuração da coleta (NewsAPI + fontes adicionais)
NEWSAPI_KEY = "SUA_CHAVE_AQUI"  # insira a chave se quiser testar
SOURCES = ["bloomberg", "reuters", "cnn", "bbc-news", "financial-post"]

query = "Ibovespa OR economia OR bolsa OR ações OR mercado financeiro"
print("🔎 Coletando notícias relacionadas a:", query)

🔎 Coletando notícias relacionadas a: Ibovespa OR economia OR bolsa OR ações OR mercado financeiro


In [5]:
# 4. Função auxiliar para coletar via NewsAPI
def get_news(source, q=query, pages=1):
    url = "https://newsapi.org/v2/everything"
    all_articles = []
    for page in range(1, pages + 1):
        params = {
            "q": q,
            "sources": source,
            "language": "pt",
            "pageSize": 50,
            "page": page,
            "apiKey": NEWSAPI_KEY,
        }
        r = requests.get(url, params=params)
        if r.status_code != 200:
            print(f"⚠️ Erro {r.status_code} em {source}")
            continue
        data = r.json().get("articles", [])
        for art in data:
            all_articles.append({
                "source": source,
                "title": art.get("title"),
                "description": art.get("description"),
                "publishedAt": art.get("publishedAt"),
            })
        time.sleep(1)
    return all_articles

In [6]:
# 5. Coletar e consolidar
news = []
for s in SOURCES:
    print(f"📡 {s}...")
    news.extend(get_news(s, pages=1))

df = pd.DataFrame(news)
print("Shape inicial:", df.shape)
display(df.head(3))

📡 bloomberg...


⚠️ Erro 401 em bloomberg
📡 reuters...


⚠️ Erro 401 em reuters
📡 cnn...


⚠️ Erro 401 em cnn
📡 bbc-news...


⚠️ Erro 401 em bbc-news
📡 financial-post...


⚠️ Erro 401 em financial-post
Shape inicial: (0, 0)


""


In [7]:
# 6. Normalização robusta + fallback e salvamento seguro
import pandas as pd
import numpy as np
import os
from datetime import datetime

def _build_text(df):
    # tenta combinar as colunas mais comuns
    title = df.get("title", pd.Series("", index=df.index)).fillna("")
    desc  = df.get("description", df.get("content", pd.Series("", index=df.index))).fillna("")
    texto = (title.astype(str) + " " + desc.astype(str)).str.strip()
    # se já existe 'texto' (pt-BR), dá preferência a ele quando for mais completo
    if "texto" in df.columns:
        base = df["texto"].astype(str).fillna("").str.strip()
        texto = np.where(base.str.len() > texto.str.len(), base, texto)
        texto = pd.Series(texto, index=df.index)
    return texto

def _normalize_api_dataframe(df):
    df = df.copy()

    # fonte: pode ser dict {"name": "..."} na NewsAPI
    if "source" in df.columns and df["source"].apply(lambda x: isinstance(x, dict)).any():
        df["fonte"] = df["source"].apply(lambda s: s.get("name") if isinstance(s, dict) else str(s))
    else:
        df["fonte"] = df.get("source", "desconhecida").astype(str)

    # data: tenta várias chaves
    date_col = next((c for c in ["publishedAt", "published_at", "date", "data", "pubDate"]
                     if c in df.columns), None)
    if date_col:
        day = pd.to_datetime(df[date_col], errors="coerce")
        # remove timezone se houver e normaliza para meia-noite
        try:
            day = day.dt.tz_localize(None)
        except Exception:
            pass
        df["day"] = day.dt.normalize()
    else:
        df["day"] = pd.NaT

    # texto
    df["texto"] = _build_text(df)

    # seleciona e limpa
    out = df[["day", "fonte", "texto"]].dropna(subset=["texto"])
    out = out[out["texto"].str.len() > 20]
    out = out.drop_duplicates()
    return out

def _load_fallback():
    # tenta reaproveitar datasets já existentes
    for fp in [
        os.path.join(RAW_PATH,  "noticias_real.csv"),
        os.path.join(PROC_PATH, "noticias_real_clean.csv"),
    ]:
        if os.path.exists(fp):
            print(f"⚠ Sem dados da API — carregando fallback: {fp}")
            base = pd.read_csv(fp)
            # normaliza para o mesmo esquema
            # mapeia data
            date_col = next((c for c in ["data", "date", "publishedAt", "pubDate"]
                             if c in base.columns), None)
            day = pd.to_datetime(base[date_col], errors="coerce") if date_col else pd.NaT
            try:
                day = day.dt.tz_localize(None)
            except Exception:
                pass
            base["day"] = day.dt.normalize() if hasattr(day, "dt") else pd.NaT

            # fonte
            if "fonte" not in base.columns:
                src_col = next((c for c in base.columns if str(c).lower() in ["fonte","source","veiculo"]), None)
                base["fonte"] = base[src_col] if src_col else "desconhecida"

            # texto
            base["texto"] = _build_text(base)

            out = base[["day", "fonte", "texto"]].dropna(subset=["texto"])
            out = out[out["texto"].str.len() > 20].drop_duplicates()
            return out
    return pd.DataFrame(columns=["day", "fonte", "texto"])

# --- fluxo principal ---
if "df" in globals() and isinstance(df, pd.DataFrame) and not df.empty:
    # veio da NewsAPI
    df = _normalize_api_dataframe(df)
else:
    # fallback com seus arquivos já coletados/limpos
    df = _load_fallback()

print("Após normalização:", df.shape)
display(df.head(5))

# Salvamento seguro
if df.empty:
    print("⚠ Dataset vazio — nada será salvo. Verifique API key/quotas ou fontes de fallback.")
else:
    out_csv  = os.path.join(PROC_PATH, "news_multisource_clean.csv")
    out_parq = os.path.join(PROC_PATH, "news_multisource_clean.parquet")
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    try:
        df.to_parquet(out_parq, index=False)
    except Exception as e:
        print("Aviso: não foi possível salvar Parquet:", e)
    print("✅ Arquivos salvos:")
    print(out_csv)
    if os.path.exists(out_parq):
        print(out_parq)

    # QC rápido
    print("Período:", df["day"].min(), "→", df["day"].max())
    print("Fontes únicas:", df["fonte"].nunique())
    print("Total de notícias:", len(df))
    display(df.sample(min(3, len(df))))
    print("Feito ✅")

⚠ Sem dados da API — carregando fallback: C:\Users\ander\OneDrive\TCC_USP\data_raw\noticias_real.csv


Após normalização: (420, 3)


,day,fonte,texto
0,2025-09-27,IGN,Esqueceu ou decidiu não comprar a Pena? O game...
1,2025-09-27,Sapo.pt,O Google Fotos poderá receber uma funcionalida...
2,2025-09-27,Observador.pt,Fogo deflagrou por volta das 11:50 deste sábad...
3,2025-09-27,Abril.com.br,Vale conferir as dicas preparadas especialment...
4,2025-09-27,Metropoles.com,Cena de Vale Tudo foi detonada nas redes socia...


✅ Arquivos salvos:


C:\Users\ander\OneDrive\TCC_USP\data_processed\news_multisource_clean.csv
C:\Users\ander\OneDrive\TCC_USP\data_processed\news_multisource_clean.parquet
Período: 2025-09-27 00:00:00 → 2025-11-17 00:00:00
Fontes únicas: 39
Total de notícias: 420


,day,fonte,texto
322,2025-11-16,Metropoles.com,"Exame trouxe temas como vacinas, clima, GNV, U..."
35,2025-09-27,InfoMoney,"Apesar do lucro sólido e do ROE elevado, fator..."
169,2025-11-09,Abril.com.br,"Em 2027, a 'herança maldita', expressão cara a..."


Feito ✅
